# 03 — 추론 + Grad-CAM

## 전체 흐름

**A. Scout** — 저장된 L8/L32/L64 txt + MTFL checkpoint → segment별 anomaly score → peak 구간  
**GPU 해제** (`clear_gpu`)  
**B. Grad-CAM** — peak 주변 clip만 **VST 백본**에 넣고 Forward → Hook → Backward (MaxChannel)

## 주의 (반드시 읽기)

- Score 산출 시 **segment 하나만 바꿔 끼우는 방식 금지** (MTFF 전체 맥락 필요) → Scout는 **전체 txt** forward
- Grad-CAM은 **Classifier 없이 VST만** 사용
- B 단계는 VST 로드 후 **한 블록 안에서** Fwd-Bwd 완료 (중간에 `clear_gpu` 하면 연산 그래프 소멸)

## 사전 준비

- `02` feature + checkpoint (`MTFL-1280.pkl` 등)
- `annotations/test.txt`
- mmaction2/mmcv (VST 로드용)


## 환경: Colab Drive 마운트


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass


## 경로·파라미터


In [ ]:
import gc
import os
import sys
from pathlib import Path

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

DRIVE_ROOT = Path("/content/drive/MyDrive/딥러닝 팀플")
WORKSPACE = DRIVE_ROOT / "04_Workspace"

sys.path.insert(0, str(WORKSPACE / "scripts"))
from config import (
    clear_gpu, DATA_UCF, MTFL_ROOT, PRETRAINED_SWIN,
    CHECKPOINTS_DIR, RESULTS_EVAL, GRADCAM_OUTPUTS, INFERENCE_CACHE,
)

sys.path.insert(0, str(MTFL_ROOT / "detection"))
sys.path.insert(0, str(MTFL_ROOT / "utils"))
os.chdir(MTFL_ROOT)

CUSTOM_ROOT = DATA_UCF
TEST_ANNO = CUSTOM_ROOT / "annotations/test.txt"
VIDEO_ROOT = CUSTOM_ROOT / "videos"
FEATURE_ROOT = CUSTOM_ROOT / "features"
MTFL_CKPT = CHECKPOINTS_DIR / "MTFL-1280.pkl"
SEG_NUM = 32
CLIP_LEN = 8
RESIZE = 224
VIDEO_INDEX = 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 헬퍼: Scout (MTFL + txt feature)


In [ ]:
def read_features_txt(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append([float(x) for x in line.strip().split()])
    return torch.tensor(rows, dtype=torch.float32)


def load_feature_triplet(rel_video):
    stem = str(Path(rel_video).with_suffix("")).replace("\\", "/")
    return (
        read_features_txt(FEATURE_ROOT / "L64" / f"{stem}.txt"),
        read_features_txt(FEATURE_ROOT / "L32" / f"{stem}.txt"),
        read_features_txt(FEATURE_ROOT / "L8" / f"{stem}.txt"),
    )


def parse_test_anno(path):
    items = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        p = line.split()
        items.append({
            "rel_video": p[0],
            "class_name": p[1],
            "num_frames": int(p[2]),
            "intervals": [int(x) for x in p[3:]],
        })
    return items


def compute_segment_scores(mtfl, lf, mf, sf):
    with torch.no_grad():
        _, _, _, _, scores = mtfl(
            lf.unsqueeze(0).to(device),
            mf.unsqueeze(0).to(device),
            sf.unsqueeze(0).to(device),
        )
    return scores.squeeze(0).squeeze(-1).detach().cpu().numpy()


## 헬퍼: Grad-CAM (VST only)


In [ ]:
MEAN = torch.tensor([0.400, 0.388, 0.372]).view(3, 1, 1, 1)
STD = torch.tensor([0.247, 0.245, 0.243]).view(3, 1, 1, 1)


def read_clip_by_frame_range(video_path, center_frame, clip_len=8, resize=224):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or center_frame + clip_len
    start = max(0, int(center_frame) - clip_len // 2)
    end = min(total, start + clip_len)
    start = max(0, end - clip_len)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    frames = []
    for _ in range(clip_len):
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (resize, resize))
        frames.append(frame)
    cap.release()
    while len(frames) < clip_len:
        frames.append(frames[-1].copy())
    arr = np.stack(frames)
    t = torch.from_numpy(arr).float() / 255.0
    t = t.permute(0, 3, 1, 2).permute(1, 0, 2, 3)
    clip = ((t - MEAN) / STD).unsqueeze(0)
    return clip, t, start, end


class VSTGradCAMHook:
    def __init__(self, layer):
        self.activation = None
        self.gradient = None
        self.h1 = layer.register_forward_hook(self._fwd)
        self.h2 = layer.register_full_backward_hook(self._bwd)

    def _fwd(self, module, inputs, output):
        self.activation = output[0] if isinstance(output, tuple) else output

    def _bwd(self, module, grad_input, grad_output):
        g = grad_output[0]
        self.gradient = g[0] if isinstance(g, tuple) else g

    def remove(self):
        self.h1.remove()
        self.h2.remove()


def run_vst_gradcam(vst, clip_tensor):
    hook = VSTGradCAMHook(vst.backbone.layers[-1])
    vst.zero_grad(set_to_none=True)
    clip_tensor = clip_tensor.to(device).requires_grad_(True)
    _ = vst.extract_feat(clip_tensor)
    act = hook.activation
    if act is None:
        raise RuntimeError("hook activation is None")
    if act.dim() == 5:
        pooled = act.mean(dim=[3, 4])
        if pooled.dim() == 4:
            pooled = pooled.mean(dim=2)
        ch_scores = pooled[0]
    else:
        ch_scores = act.view(act.size(0), act.size(1), -1).mean(-1)[0]
    max_idx = int(ch_scores.argmax().item())
    target = ch_scores[max_idx]
    target.backward()
    grad = hook.gradient
    weights = grad.mean(dim=[2, 3, 4], keepdim=True) if grad.dim() == 5 else grad.mean(dim=-1, keepdim=True)
    cam = torch.relu((weights * act).sum(dim=1))
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-6)
    hook.remove()
    return cam.detach().cpu().numpy(), max_idx


## A. Scout (실행)


In [ ]:
from model import Model as MTFLModel

items = parse_test_anno(TEST_ANNO)
item = items[VIDEO_INDEX]
rel_video = item["rel_video"]
video_path = VIDEO_ROOT / Path(rel_video)
stem = Path(rel_video).stem

lf, mf, sf = load_feature_triplet(rel_video)
mtfl = MTFLModel(feature_dim=1024, batch_size=1, seg_num=SEG_NUM).to(device).eval()
mtfl.load_state_dict(torch.load(MTFL_CKPT, map_location=device))
segment_scores = compute_segment_scores(mtfl, lf, mf, sf)
target_segment = int(np.argmax(segment_scores))
num_frames = item["num_frames"]
center_frame = int((target_segment + 0.5) * num_frames / SEG_NUM)

scores_path = RESULTS_EVAL / "scores" / f"{stem}_scores.npy"
scores_path.parent.mkdir(parents=True, exist_ok=True)
frame_scores = np.zeros(num_frames, dtype=np.float32)
for seg_i, sc in enumerate(segment_scores):
    s = int(seg_i * num_frames / SEG_NUM)
    e = int((seg_i + 1) * num_frames / SEG_NUM)
    frame_scores[s:e] = sc
np.save(scores_path, frame_scores)
peak_dir = INFERENCE_CACHE / stem
peak_dir.mkdir(parents=True, exist_ok=True)
torch.save(
    {"target_segment": target_segment, "center_frame": center_frame, "rel_video": rel_video},
    peak_dir / "peak_meta.pt",
)

del mtfl
clear_gpu()
print("peak segment:", target_segment, "center_frame:", center_frame)
print("saved:", scores_path)


### A 확인: segment score 곡선


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(segment_scores, marker="o", label="segment score")
plt.axvline(target_segment, color="r", linestyle="--", label=f"peak seg {target_segment}")
plt.title(rel_video)
plt.xlabel("segment index")
plt.ylabel("score")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## A 종료: GPU 메모리 해제


In [ ]:
# Scout 변수는 유지, MTFL만 해제됨. B 전에 한 번 더 비울 수 있습니다.
gc.collect()
clear_gpu()
print("ready for Grad-CAM (VST)")


## B. Grad-CAM (실행) — VST Forward-Backward 연속


In [ ]:
from feature_extractor import load_VST

meta = torch.load(INFERENCE_CACHE / stem / "peak_meta.pt", map_location="cpu")
center_frame = int(meta["center_frame"])
clip_t, display_t, clip_start, clip_end = read_clip_by_frame_range(
    video_path, center_frame, CLIP_LEN, RESIZE
)

vst = load_VST(str(PRETRAINED_SWIN), device).eval()
cam_np, max_ch = run_vst_gradcam(vst, clip_t)
del vst
clear_gpu()

mid = display_t.size(0) // 2
frame = display_t[mid].numpy().transpose(1, 2, 0)
frame = np.uint8(np.clip(frame * 255, 0, 255))
cam_2d = cam_np[0, cam_np.shape[1] // 2] if cam_np.ndim == 3 else cam_np[0]
cam_2d = cv2.resize(cam_2d, (frame.shape[1], frame.shape[0]))
heatmap = cv2.applyColorMap(np.uint8(255 * cam_2d), cv2.COLORMAP_JET)
overlay = cv2.cvtColor(
    cv2.addWeighted(frame[:, :, ::-1], 0.55, heatmap, 0.45, 0), cv2.COLOR_BGR2RGB
)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(frame)
ax[0].set_title(f"frame {clip_start + mid}")
ax[0].axis("off")
ax[1].imshow(overlay)
ax[1].set_title(f"VST Grad-CAM (ch={max_ch})")
ax[1].axis("off")
plt.suptitle(rel_video)
plt.show()


### B 확인: Grad-CAM 영상 저장


In [ ]:
GRADCAM_OUTPUTS.mkdir(parents=True, exist_ok=True)
out_mp4 = GRADCAM_OUTPUTS / f"{stem}_gradcam.mp4"
h, w = overlay.shape[:2]
writer = cv2.VideoWriter(str(out_mp4), cv2.VideoWriter_fourcc(*"mp4v"), 4.0, (w, h))
for _ in range(8):
    writer.write(cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
writer.release()
print("saved:", out_mp4)
